In [10]:
from client import *
from common_imports import *
from collections.abc import Awaitable, Callable
from agent_framework import (AgentMiddleware,
                             AgentContext, 
                             ChatContext, 
                             FunctionInvocationContext, 
                             FunctionMiddleware,
                            ChatMiddleware,
                            AgentResponse,
                            Message)
import time

In [11]:
class BlockingAgentMiddleware(AgentMiddleware):
    """Agent middleware that blocks requests containing forbidden words."""

    def __init__(self, blocked_words: list[str]) -> None:
        """Initialize with a list of words that should be blocked."""
        self.blocked_words = blocked_words

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Check messages for blocked content and terminate if found."""
        last_message = context.messages[-1] if context.messages else None
        if last_message and last_message.text:
            for word in self.blocked_words:
                if word.lower() in last_message.text.lower():
                    logger.warning(f"[❌ Blocking][ Agent Middleware] Request blocked: contains '{word}'")
                    context.terminate = True
                    context.result = AgentResponse(
                        messages=[
                            Message(role="assistant", contents=[f"Sorry, I can't process requests about '{word}'."])
                        ]
                    )
                    return

        await call_next()

In [12]:
class MessageCountChatMiddleware(ChatMiddleware):
    """Chat middleware that tracks the total number of messages sent to the AI."""

    def __init__(self) -> None:
        """Initialize the message counter."""
        self.total_messages = 0

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """Count messages and log the running total."""
        self.total_messages += len(context.messages)
        logger.info(
            "[🔢 Message Count][ Chat Middleware] Messages in this request: %s, total so far: %s",
            len(context.messages),
            self.total_messages,
        )

        await call_next()

        logger.info("[🔢 Message Count][ Chat Middleware] Chat response received")


In [13]:
async def logging_function_middleware(
    context: FunctionInvocationContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Function middleware that logs function calls and results."""
    logger.info(f"[🪵 Logging][ Function Middleware] Calling {context.function.name} with args: {context.arguments}")

    await call_next()

    logger.info(f"[🪵 Logging][ Function Middleware] {context.function.name} returned: {context.result}")


async def logging_chat_middleware(
    context: ChatContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Chat middleware that logs AI interactions."""
    logger.info(f"[💬 Logging][ Chat Middleware] Sending {len(context.messages)} messages to AI")

    await call_next()

    logger.info("[💬 Logging][ Chat Middleware] AI response received")

In [14]:
async def timing_agent_middleware(
    context: AgentContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Agent middleware that logs execution timing."""
    start = time.perf_counter()
    logger.info("[⏲️ Timing][ Agent Middleware] Starting agent execution")

    await call_next()

    elapsed = time.perf_counter() - start
    logger.info(f"[⏲️ Timing][ Agent Middleware] Execution completed in {elapsed:.2f}s")

In [15]:
def get_weather(
    city: Annotated[str, Field(description="The city to get the weather for.")],
) -> dict:
    """Returns weather data for a given city, a dictionary with temperature and description."""
    logger.info(f"Getting weather for {city}")
    if random.random() < 0.05:
        return {
            "temperature": 72,
            "description": "Sunny",
        }
    else:
        return {
            "temperature": 60,
            "description": "Rainy",
        }


def get_activities(
    city: Annotated[str, Field(description="The city to get activities for.")],
    date: Annotated[str, Field(description="The date to get activities for in format YYYY-MM-DD.")],
) -> list[dict]:
    """Returns a list of activities for a given city and date."""
    logger.info(f"Getting activities for {city} on {date}")
    return [
        {"name": "Hiking", "location": city},
        {"name": "Beach", "location": city},
        {"name": "Museum", "location": city},
    ]


def get_current_date() -> str:
    """Gets the current date from the system and returns as a string in format YYYY-MM-DD."""
    logger.info("Getting current date")
    return datetime.now().strftime("%Y-%m-%d")


In [16]:
message_count_middleware = MessageCountChatMiddleware()
blocking_middleware = BlockingAgentMiddleware(blocked_words=["nuclear", "classified"])

In [17]:
agent = Agent(
    client=client,
    instructions=(
        "You help users plan their weekends and choose the best activities for the given weather. "
        "If an activity would be unpleasant in weather, don't suggest it. Include date of the weekend in response."
    ),
    tools=[get_weather, get_activities, get_current_date],
    middleware=[
        # Agent-level middleware applied to ALL runs
        logging_chat_middleware,
        logging_function_middleware,
        timing_agent_middleware,
        blocking_middleware,    
        message_count_middleware
    ],
)

In [18]:
response = await agent.run("Tell me about nuclear physics.")
print(response)

[⏲️ Timing][ Agent Middleware] Starting agent execution
[❌ Blocking][ Agent Middleware] Request blocked: contains 'nuclear'
[⏲️ Timing][ Agent Middleware] Execution completed in 0.00s


Sorry, I can't process requests about 'nuclear'.
